# 🚀 GroceryGOD // Automated Parallel Uplink
This notebook scrapes Shwapno, Chaldal, Meena Bazar, and Othoba **in parallel**, aggregates the data, and pushes it to GitHub Pages.

### 🛠️ Setup Instructions:
1. Go to **Add-ons -> Secrets** in Kaggle.
2. Add `GITHUB_PAT` (Your GitHub Token).
3. Add `TELEGRAM_TOKEN` (Optional).
4. Add `TELEGRAM_CHAT_ID` (Optional).
5. Run all cells.

In [ ]:
# ============================================================
# CELL 0 ▸ INSTALL DEPENDENCIES
# ============================================================
import subprocess, sys, time

_cell_start = time.time()
print('=' * 60)
print('📦  [CELL 0] Installing Dependencies')
print('=' * 60)

def _run(cmd, label):
    print(f'  ⏳  {label}...')
    t = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.time() - t
    if r.returncode == 0:
        print(f'  ✅  {label} done  ({elapsed:.1f}s)')
    else:
        print(f'  ❌  {label} FAILED  ({elapsed:.1f}s)')
        print(r.stderr[-800:] if r.stderr else '')
    return r.returncode

_run('pip install -q playwright sqlalchemy aiosqlite requests', 'pip install packages')
_run('playwright install chromium --with-deps', 'playwright install chromium')
_run('apt-get install -y -q sqlite3', 'apt-get install sqlite3')

print(f'\n✅  CELL 0 complete  — {time.time()-_cell_start:.1f}s')
print('=' * 60)

In [ ]:
# ============================================================
# CELL 1 ▸ TELEGRAM NOTIFIER + GLOBAL LOGGER SETUP
# ============================================================
import os, io, glob, time, logging, traceback, requests, concurrent.futures
from datetime import datetime, timedelta
from pathlib import Path
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

try:
    TELEGRAM_BOT_TOKEN = user_secrets.get_secret("TELEGRAM_TOKEN")
    TELEGRAM_CHAT_ID   = user_secrets.get_secret("TELEGRAM_CHAT_ID")
except:
    TELEGRAM_BOT_TOKEN = None
    TELEGRAM_CHAT_ID   = None

TG_API = f'https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}' if TELEGRAM_BOT_TOKEN else None
PIPELINE_START = time.time()
STEP_LOG = []

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler('/tmp/grocerygod_run.log', mode='w')]
)
log = logging.getLogger('GroceryGOD')

def _fmt_dur(seconds: float) -> str:
    return str(timedelta(seconds=int(seconds)))

def tg_send(text: str, silent: bool = False) -> bool:
    if not TG_API or not TELEGRAM_CHAT_ID: return False
    try:
        r = requests.post(f'{TG_API}/sendMessage', json={'chat_id': TELEGRAM_CHAT_ID, 'text': text, 'parse_mode': 'HTML', 'disable_notification': silent}, timeout=15)
        return r.ok
    except: return False

class Step:
    def __init__(self, name: str, emoji: str = '⚙️', notify: bool = True):
        self.name, self.emoji, self.notify = name, emoji, notify
        self._t0 = None
    def __enter__(self):
        self._t0 = time.time()
        log.info(f'{self.emoji}  [{self.name}] — STARTED')
        if self.notify: tg_send(f'{self.emoji} <b>{self.name}</b> — started', silent=True)
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.time() - self._t0
        if exc_type is None:
            log.info(f'✅  [{self.name}] — OK ({_fmt_dur(elapsed)})')
            STEP_LOG.append(('✅', self.name, elapsed))
            if self.notify: tg_send(f'✅ <b>{self.name}</b> — complete ({_fmt_dur(elapsed)})')
        else:
            tb_str = traceback.format_exc()
            log.error(f'❌  [{self.name}] — FAILED\n{tb_str}')
            STEP_LOG.append(('❌', self.name, elapsed))
            if self.notify: tg_send(f'❌ <b>{self.name}</b> — FAILED\n<pre>{tb_str[-1000:]}</pre>')
            return False
        return True

tg_send(f'🚀 <b>GroceryGOD Parallel Pipeline — STARTED</b>')
print('\n✅  Telegram notifier ready.')

In [ ]:
# ============================================================
# CELL 2 ▸ CONFIGURATION & GIT SETUP
# ============================================================
with Step('Configuration & Git Setup', '⚙️'):
    PAT = user_secrets.get_secret('GITHUB_PAT')
    REPO_URL = f'https://{PAT}@github.com/ranx-x/GroceryGOD.git'
    subprocess.run('git config --global user.email "ran.ragibahnafnehal2@gmail.com"', shell=True)
    subprocess.run('git config --global user.name "ranx-x"', shell=True)
    if not os.path.exists('GroceryGOD'):
        subprocess.run(f'git clone {REPO_URL}', shell=True, check=True)
    os.chdir('GroceryGOD')
    log.info(f'📂 Working directory: {os.getcwd()}')

In [ ]:
# ============================================================
# CELL 3 ▸ EXECUTE SCRAPERS (PARALLEL)
# ============================================================
def run_scraper(scraper_info):
    label, path = scraper_info
    log.info(f'▶️ Starting {label}...')
    t0 = time.time()
    # Note: Use full path to ensure subprocess finds the script
    full_path = os.path.join(os.getcwd(), path)
    res = subprocess.run('python scraper.py', cwd=full_path, shell=True, capture_output=True, text=True, timeout=3600)
    elapsed = time.time() - t0
    status = '✅' if res.returncode == 0 else '❌'
    log.info(f'   {status} {label} finished in {_fmt_dur(elapsed)}')
    tg_send(f'{status} <b>{label}</b> — {_fmt_dur(elapsed)}', silent=True)
    return label, res.returncode == 0

with Step('Parallel Market Scrapers', '🛰️'):
    scrapers = [
        ('🛒 Shwapno', 'swapnoTRACKER'),
        ('🟢 Chaldal', 'PRICETRACKER'),
        ('🏪 Meena Bazar', 'MEENAtracker/backend'),
        ('🔵 Othoba', 'othobaTRACKER/backend')
    ]
    
    log.info(f'🚀 Launching {len(scrapers)} scrapers in parallel pool...')
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(run_scraper, scrapers))
    
    for label, success in results:
        if not success: log.warning(f'⚠️ {label} scraper reported failure.')

In [ ]:
# ============================================================
# CELL 4 ▸ RUN AGGREGATOR
# ============================================================
with Step('GODdata Aggregator', '🧬'):
    res = subprocess.run('python aggregator.py', shell=True, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f'Aggregator failed: {res.stdout[-500:]}')
    log.info('✅ Aggregation Complete.')

In [ ]:
# ============================================================
# CELL 5 ▸ PUSH TO GITHUB
# ============================================================
with Step('GitHub Push', '🚀'):
    subprocess.run('git add .', shell=True)
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    subprocess.run(f'git commit -m "🤖 Automated Market Update (Parallel): {now}"', shell=True)
    subprocess.run('git push origin master --force', shell=True, check=True)
    log.info('✅ Push successful!')
    tg_send(f'🚀 <b>GitHub Push Successful!</b>\n🌐 Live at https://ranx-x.github.io/GroceryGOD')